In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

from scipy.stats import norm
from scipy.optimize import minimize

In [2]:
# ----------------------------- config ---------------------------------------
EPS = 1e-6
MIN_SAMPLES = 10
N_DRESS = 50
RNG_SEED = 42


years_train = np.arange(1959, 2000)
years_verif = np.arange(2000, 2021)
T_CENTER = float(years_train.mean())   # e.g., 1979.0 for 1959..1999

# all Gaussian-distributed variables to post-process
vars_keep = [
    # 4-parameter clearly better
    "Fairbanks_T2_1d_max",
    "Fairbanks_T2_1h_max",
    "Fairbanks_T2_30d_max",
    "Fairbanks_T2_7d_max",
    "Fairbanks_simple_T2_1d_max",
    "Fairbanks_simple_T2_1h_max",
    "Fairbanks_simple_T2_30d_max",
    "Fort_Bragg_SPEI_09_min",          # ACC tied, CRPS clearly better
    "Fort_Bragg_simple_T2_1d_mean",
    "Guam_simple_T2_1d_mean",
    "Guam_simple_T2_1d_min",
    "Guam_simple_T2_1h_min",
    "Pituffik_simple_T2_1d_max",       # ACC both ~0, CRPS 1.22 vs 1.86
    "Pituffik_simple_T2_1d_mean",
    "Pituffik_simple_T2_1h_max",       # ACC both ~0, CRPS 1.22 vs 3.08
    "Pituffik_simple_T2_30d_max",
    "Yuma_PG_SPEI_24_mean",            # ACC tied, CRPS better
    "Yuma_PG_SPEI_24_min",             # ACC both ~0, CRPS better
    "Yuma_PG_simple_T2_1d_max",
    "Yuma_PG_simple_T2_1h_max",
    "Yuma_PG_simple_T2_1h_min",        # ACC both ~0, CRPS better
    "Yuma_PG_simple_T2_30d_max",
    # No clear difference
    "Fairbanks_simple_T2_1d_mean",
    "Fairbanks_simple_T2_1d_min",
    "Fairbanks_simple_T2_1h_min",
    "Fort_Bragg_simple_T2_1d_min",
    "Fort_Bragg_simple_T2_1h_min",
    "Guam_simple_T2_1d_max",
    "Guam_simple_T2_30d_max",
    "Yuma_PG_days_above_999p",
]

data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
in_input  = os.path.join(data_dir, 'STN_CESM_ALL_20260604.zarr')
in_target = os.path.join(data_dir, 'STN_ERA5_ALL_20260604.zarr')

# ----------------------------- load -----------------------------------------
ds_input = xr.open_zarr(in_input)[vars_keep].drop_vars('valid_year', errors='ignore')
ds_input_train = ds_input.sel(init_year=years_train)
ds_input_verif = ds_input.sel(init_year=years_verif)
 
ds_target = xr.open_zarr(in_target)[vars_keep]
 
n_member = ds_input_train.sizes['member']
n_lead   = ds_input_train.sizes['lead_year']
n_train  = ds_input_train.sizes['init_year']
n_verif  = ds_input_verif.sizes['init_year']
 
lead_years        = ds_input_train['lead_year'].values
init_years_train  = ds_input_train['init_year'].values
init_years_verif  = ds_input_verif['init_year'].values

In [3]:
# def fit_emos(xbar, sx2, y):
#     """OLS-only EMOS parameter estimation (no CRPS minimisation).

#     Two-stage estimator:
#       Stage 1: y = a + b * xbar         -> (a, b) by OLS on the mean
#       Stage 2: (y - mu)^2 = c^2 + d^2 * sx2  -> (c^2, d^2) by OLS on the variance

#     Faster and deterministic than CRPS-minimised EMOS, but generally yields
#     slightly larger CRPS because the two stages are optimised separately
#     rather than jointly against a single proper scoring rule.

#     Returns (params, success) with the same shape as the L-BFGS-B version:
#     params = [a, b, c, d], where the predictive variance is c^2 + d^2 * sx2.
#     """
#     valid = np.isfinite(xbar) & np.isfinite(sx2) & np.isfinite(y)
#     if valid.sum() < MIN_SAMPLES:
#         return np.full(4, np.nan), False
#     xbar_v, sx2_v, y_v = xbar[valid], sx2[valid], y[valid]

#     # --- Stage 1: mean regression -----------------------------------------
#     try:
#         A = np.column_stack([np.ones_like(xbar_v), xbar_v])
#         ab, *_ = np.linalg.lstsq(A, y_v, rcond=None)
#         a_hat, b_hat = float(ab[0]), float(ab[1])
#     except np.linalg.LinAlgError:
#         return np.full(4, np.nan), False

#     # Enforce b >= 0; if OLS slope is negative, collapse to climatology
#     if b_hat < 0.0:
#         b_hat = 0.0
#         a_hat = float(np.mean(y_v))

#     # --- Stage 2: variance regression -------------------------------------
#     mu_hat = a_hat + b_hat * xbar_v
#     resid2 = (y_v - mu_hat) ** 2
#     try:
#         B = np.column_stack([np.ones_like(sx2_v), sx2_v])
#         cd2, *_ = np.linalg.lstsq(B, resid2, rcond=None)
#         c2_hat, d2_hat = float(cd2[0]), float(cd2[1])
#     except np.linalg.LinAlgError:
#         # Fall back to homoscedastic estimate (no spread-skill relationship)
#         c2_hat = float(np.mean(resid2))
#         d2_hat = 0.0

#     # Enforce non-negativity. If the OLS intercept c^2 is negative, the
#     # spread-skill regression has a steep positive slope and a negative
#     # offset; clamping c^2 to EPS keeps the predictive variance well-defined
#     # without distorting d^2 much in the typical sx2 range.
#     c2_hat = max(c2_hat, EPS)
#     d2_hat = max(d2_hat, 0.0)

#     params = np.array([a_hat, b_hat, np.sqrt(c2_hat), np.sqrt(d2_hat)])
#     return params, True

In [4]:
# ============================================================================
# Gaussian EMOS
# ============================================================================
def crps_gaussian(mu, sigma, y):
    """Closed-form CRPS for N(mu, sigma^2)."""
    sigma = np.maximum(sigma, EPS)
    z = (y - mu) / sigma
    return sigma * (z * (2.0 * norm.cdf(z) - 1.0)
                    + 2.0 * norm.pdf(z)
                    - 1.0 / np.sqrt(np.pi))
 
def emos_objective(params, xbar, sx2, y):
    a, b, c, d = params
    mu = a + b * xbar
    sigma2 = c * c + d * d * sx2
    sigma = np.sqrt(np.maximum(sigma2, EPS * EPS))
    return float(np.mean(crps_gaussian(mu, sigma, y)))

def fit_emos_standardized(xbar, sx2, y, gamma_min=0.0):
    valid = np.isfinite(xbar) & np.isfinite(sx2) & np.isfinite(y)
    if valid.sum() < MIN_SAMPLES:
        return np.full(4, np.nan), False
    xbar_v, sx2_v, y_v = xbar[valid], sx2[valid], y[valid]

    # Training-period scalings
    xbar_mean = float(np.mean(xbar_v))
    xbar_std  = float(np.std(xbar_v, ddof=1))
    y_mean    = float(np.mean(y_v))
    y_std     = float(np.std(y_v, ddof=1))

    # Standardized predictor and target
    xs = (xbar_v - xbar_mean) / max(xbar_std, EPS)
    ys = (y_v    - y_mean)    / max(y_std,    EPS)
    sx2_norm = sx2_v / max(xbar_std ** 2, EPS)

    # OLS init: gamma = corr(xs, ys), tau in [0, 1]-ish for spread coupling
    gamma_0 = float(np.clip(np.corrcoef(xs, ys)[0, 1], gamma_min, 1.0))
    resid2 = (ys - gamma_0 * xs) ** 2
    try:
        B = np.column_stack([np.ones_like(sx2_norm), sx2_norm])
        cd2, *_ = np.linalg.lstsq(B, resid2, rcond=None)
        c2_0 = max(float(cd2[0]), EPS)
        d2_0 = max(float(cd2[1]), EPS)
    except Exception:
        c2_0, d2_0 = float(np.var(ys)) + EPS, 0.0

    def obj(p):
        gamma, c, d = p
        mu_s = gamma * xs
        sigma_s = np.sqrt(np.maximum(c * c + d * d * sx2_norm, EPS * EPS))
        return float(np.mean(crps_gaussian(mu_s, sigma_s, ys)))

    x0 = np.array([gamma_0, np.sqrt(c2_0), np.sqrt(d2_0)])
    bounds = [(gamma_min, 1.0), (None, None), (None, None)]
    res = minimize(obj, x0, method='L-BFGS-B', bounds=bounds)
    gamma_hat, c_hat, d_hat = res.x

    # Convert back to (a, b, c, d) in original units for downstream compatibility
    b_hat = gamma_hat * y_std / max(xbar_std, EPS)
    a_hat = y_mean - b_hat * xbar_mean
    # Spread coefficients: c, d operate on standardized variance, scale back to y units
    c_phys = c_hat * y_std
    d_phys = d_hat * y_std / max(xbar_std, EPS)

    return np.array([a_hat, b_hat, c_phys, d_phys]), bool(res.success)

    
def lookup_target(y_full, valid_years_needed, valid_years_avail):
    out = np.full(len(valid_years_needed), np.nan, dtype=np.float64)
    in_mask = np.isin(valid_years_needed, valid_years_avail)
    if in_mask.any():
        out[in_mask] = y_full.sel(valid_year=valid_years_needed[in_mask]).values
    return out



In [5]:
# ============================================================================
# Step 1: Training -- fit EMOS per (variable, lead_year)
# ============================================================================
params_dict = {v: np.full((n_lead, 4), np.nan) for v in vars_keep}
fit_flags   = {v: np.zeros(n_lead, dtype=bool) for v in vars_keep}
 
for var in tqdm(vars_keep, desc='EMOS training '):
    x_train = ds_input_train[var].values
    y_full  = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    for il, lead in enumerate(lead_years):
        xbar = x_train[:, il, :].mean(axis=1)
        sx2  = x_train[:, il, :].var(axis=1, ddof=1)
        valid_yrs_needed = init_years_train + int(lead)
        y_arr = lookup_target(y_full, valid_yrs_needed, valid_years_avail)
        params, success = fit_emos_standardized(xbar, sx2, y_arr, gamma_min=0.1)
        params_dict[var][il, :] = params
        fit_flags[var][il]      = success

EMOS training : 100%|██████████| 30/30 [00:28<00:00,  1.05it/s]


In [6]:
# for il in range(n_lead):
#     xb = ds_input_train[var].values[:, il, :].mean(axis=1)
#     yv = lookup_target(ds_target[var], init_years_train + int(lead_years[il]),
#                        ds_target[var]['valid_year'].values)
#     m = np.isfinite(xb) & np.isfinite(yv)
#     if m.sum() < 5:
#         print(f"lead {il}: too few pairs")
#         continue
#     print(f"lead {il}:  mean(xbar)={xb[m].mean():.2f}  std(xbar)={xb[m].std():.3f}  "
#           f"mean(y)={yv[m].mean():.2f}  std(y)={yv[m].std():.3f}  "
#           f"corr={np.corrcoef(xb[m], yv[m])[0,1]:+.3f}")

In [7]:
# ============================================================================
# Step 2: Calibration on verification data
# Step 3: Ensemble dressing (50 members)
# ============================================================================
rng = np.random.default_rng(RNG_SEED)
 
calib_data = {}
dress_data = {}
 
for var in tqdm(vars_keep, desc='EMOS calibration'):
    x_verif = ds_input_verif[var].values
    P = params_dict[var]
 
    mu_arr    = np.full((n_verif, n_lead), np.nan)
    sigma_arr = np.full((n_verif, n_lead), np.nan)
 
    for il in range(n_lead):
        a, b, c, d = P[il]
        if not np.isfinite(a):
            continue
        xbar = x_verif[:, il, :].mean(axis=1)
        sx2  = x_verif[:, il, :].var(axis=1, ddof=1)
        mu_arr[:, il]    = a + b * xbar
        sigma_arr[:, il] = np.sqrt(np.maximum(c * c + d * d * sx2, EPS * EPS))
        
    calib_data[var + '_mu']    = (('init_year', 'lead_year'), mu_arr)
    calib_data[var + '_sigma'] = (('init_year', 'lead_year'), sigma_arr)
 
    noise = rng.standard_normal((n_verif, n_lead, N_DRESS))
    dressed = mu_arr[:, :, None] + sigma_arr[:, :, None] * noise
    dress_data[var] = (('init_year', 'lead_year', 'member'), dressed)
 
ds_calib = xr.Dataset(
    calib_data,
    coords={'init_year': init_years_verif, 'lead_year': lead_years},
)
 
ds_dress = xr.Dataset(
    dress_data,
    coords={
        'init_year': init_years_verif,
        'lead_year': lead_years,
        'member':    np.arange(N_DRESS),
    },
)
 
param_arr = np.stack([params_dict[v] for v in vars_keep], axis=0)
ds_params = xr.Dataset(
    {'emos_params': (('variable', 'lead_year', 'param'), param_arr),
     'fit_success': (('variable', 'lead_year'),
                     np.stack([fit_flags[v] for v in vars_keep], axis=0))},
    coords={
        'variable':  vars_keep,
        'lead_year': lead_years,
        'param':     ['a', 'b', 'c', 'd'],
    },
)

EMOS calibration: 100%|██████████| 30/30 [00:09<00:00,  3.01it/s]


In [14]:
# ============================================================================
# Step 4: Verification (CRPS and ACC)
# ============================================================================
def crps_ensemble(ens, y):
    out = np.full(ens.shape[0], np.nan)
    finite_ens = np.all(np.isfinite(ens), axis=1)
    mask = finite_ens & np.isfinite(y)
    if not mask.any():
        return out
    e = ens[mask]
    yv = y[mask]
    term1 = np.mean(np.abs(e - yv[:, None]), axis=1)
    diff = np.abs(e[:, :, None] - e[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    out[mask] = term1 - term2
    return out
 
# ---- (b) per-system climatology --------------------------------------------
def anomaly_correlation(fc, obs, fc_clim=None, obs_clim=None):
    """ACC with optionally separate forecast and observation climatologies.
    If clims are None, each is taken as the mean of its own series over the
    (finite) sample -- equivalent to Pearson r."""
    m = np.isfinite(fc) & np.isfinite(obs)
    if m.sum() < 2:
        return np.nan
    fc_m, obs_m = fc[m], obs[m]
    if fc_clim is None:
        fc_clim = float(np.mean(fc_m))
    if obs_clim is None:
        obs_clim = float(np.mean(obs_m))
    fa = fc_m  - fc_clim
    oa = obs_m - obs_clim
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan
 
verif_rows = []
for var in tqdm(vars_keep, desc='Verification   '):
    y_full = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    x_raw   = ds_input_verif[var].values
    x_dress = ds_dress[var].values
    mu      = ds_calib[var + '_mu'].values
    sigma   = ds_calib[var + '_sigma'].values
 
    for il, lead in enumerate(lead_years):
        valid_yrs = init_years_verif + int(lead)
        y_arr = lookup_target(y_full, valid_yrs, valid_years_avail)
 
        # CRPS
        crps_raw    = crps_ensemble(x_raw[:, il, :],   y_arr)
        crps_dress  = crps_ensemble(x_dress[:, il, :], y_arr)
        crps_closed = crps_gaussian(mu[:, il], sigma[:, il], y_arr)
 
        # ACC with per-system climatology (each centered on its own mean)
        raw_mean   = x_raw[:, il, :].mean(axis=1)
        dress_mean = x_dress[:, il, :].mean(axis=1)
        calib_mean = mu[:, il]
 
        acc_raw   = anomaly_correlation(raw_mean,   y_arr)
        acc_dress = anomaly_correlation(dress_mean, y_arr)
        acc_calib = anomaly_correlation(calib_mean, y_arr)
        # Per-system clims (recorded for transparency)
        fin_y      = np.isfinite(y_arr)
        clim_obs   = float(np.nanmean(y_arr))      if fin_y.any() else np.nan
        clim_raw   = float(np.nanmean(raw_mean))   if np.isfinite(raw_mean).any()   else np.nan
        clim_calib = float(np.nanmean(calib_mean)) if np.isfinite(calib_mean).any() else np.nan
 
        mean_raw    = float(np.nanmean(crps_raw))
        mean_dress  = float(np.nanmean(crps_dress))
        mean_closed = float(np.nanmean(crps_closed))
        crpss = (1.0 - mean_closed / mean_raw) if (mean_raw and mean_raw > 0) else np.nan
 
        verif_rows.append({
            'variable':    var,
            'lead_year':   int(lead),
            'n_eff':       int(fin_y.sum()),
            'clim_obs':    clim_obs,
            'clim_raw':    clim_raw,
            'clim_calib':  clim_calib,
            'CRPS_raw':    mean_raw,
            'CRPS_dress':  mean_dress,
            'CRPS_closed': mean_closed,
            'CRPSS':       crpss,
            'ACC_raw':     acc_raw,
            'ACC_dress':   acc_dress,
            'ACC_calib':   acc_calib,
            'b':           float(params_dict[var][il, 1]),
            'fit_ok':      bool(fit_flags[var][il]),
        })
 
df_verif = pd.DataFrame(verif_rows)

Verification   : 100%|██████████| 30/30 [00:12<00:00,  2.32it/s]


In [9]:
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/EMOS/'
ds_dress.to_zarr(save_dir+'Dress_4param.zarr', mode='w')
ds_calib.to_zarr(save_dir+'Calib_4param.zarr', mode='w')

In [10]:
print("\n=== EMOS with 4 params ===")
summary = df_verif.groupby('variable')[['CRPS_raw', 'CRPS_dress', 'CRPSS', 'ACC_raw', 'ACC_calib']].mean().round(3)
print(summary.to_string())


=== EMOS with 4 params ===
                              CRPS_raw  CRPS_dress  CRPSS  ACC_raw  ACC_calib
variable                                                                     
Fairbanks_T2_1d_max              4.477       0.925  0.800    0.228      0.228
Fairbanks_T2_1h_max              4.211       1.085  0.746    0.088      0.088
Fairbanks_T2_30d_max             4.934       0.827  0.834    0.139      0.139
Fairbanks_T2_7d_max              4.532       0.840  0.815    0.125      0.125
Fairbanks_simple_T2_1d_max       4.477       0.915  0.800    0.228      0.228
Fairbanks_simple_T2_1d_mean      1.698       0.673  0.584    0.308      0.308
Fairbanks_simple_T2_1d_min       2.392       2.656 -0.065    0.227      0.227
Fairbanks_simple_T2_1h_max       4.211       1.109  0.746    0.088      0.088
Fairbanks_simple_T2_1h_min       2.547       2.841 -0.078    0.176      0.176
Fairbanks_simple_T2_30d_max      4.934       0.838  0.834    0.139      0.139
Fort_Bragg_SPEI_09_min           0.9